In [7]:
import os
import glob
import tqdm
import torchaudio
import pandas as pd

In [2]:
def get_duration(audio_path):
    info = torchaudio.info(audio_path)
    duration = info.num_frames / info.sample_rate
    return duration

In [11]:
dorado_wav_files = glob.glob(f'./data/**/*.wav', recursive=True)
native_speaker_wav_root = '/data/mtseng/voice_datasets/LibriTTS/data'
output_dir = f'./parallel_data/golden_speakers'
pairs = []
error_files = []
for dorado_wav_file in tqdm.tqdm(dorado_wav_files):
    golden_speaker_wav_file = dorado_wav_file.replace(f'./data', native_speaker_wav_root)
    assert os.path.exists(golden_speaker_wav_file), f'{golden_speaker_wav_file} does not exist'

    nonnative_duration = get_duration(dorado_wav_file)
    golden_speaker_duration = get_duration(golden_speaker_wav_file)
    if abs(nonnative_duration - golden_speaker_duration) > 0.1:
        error_files.append((dorado_wav_file, nonnative_duration, "duration mismatch with golden speaker wav file"))
        continue
    pairs.append(
        {
            "src_wav": os.path.abspath(golden_speaker_wav_file),
            "tgt_wav": os.path.abspath(dorado_wav_file)
        }
    )
pairs_df = pd.DataFrame(pairs)

  0%|          | 0/54478 [00:00<?, ?it/s]/tmp/ipykernel_284043/1964967775.py:2: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info = torchaudio.info(audio_path)
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/torchaudio/_backend/sox.py:24: UserWarning: torchaudio._backend.common.AudioMetaData has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It 

In [12]:
print(f"Total {len(dorado_wav_files)} wav files, {len(error_files)} files have errors.")

Total 54478 wav files, 0 files have errors.


In [13]:
num_short_files = sum(1 for _, duration, _ in error_files if duration < 2.0)
print(f"{num_short_files} files have duration shorter than 2 seconds.")
num_long_files = sum(1 for _, duration, _ in error_files if duration >= 2.0)
print(f"{num_long_files} files have duration longer than or equal to 2 seconds.")

0 files have duration shorter than 2 seconds.
0 files have duration longer than or equal to 2 seconds.


In [14]:
pairs_df.to_csv('./metadata/american_to_spanish.csv', index=False)